In [1]:
%pwd

's:\\Projects\\Text_Summariser\\reseach'

In [2]:
import os
os.chdir("../")
%pwd

's:\\Projects\\Text_Summariser'

In [18]:
# define entities
# from config.yaml
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen = True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: str
    MAX_INPUT_LENGTH: int
    MAX_TARGET_LENGTH: int
    

In [12]:
from Text_summariser.constant import *
from Text_summariser.utils.common import read_yaml, create_directories

In [13]:
# 4 Update configuration manager

class configurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,     # Access to constants
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath) # read all config and params yaml files
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root]) # same upto here for most pipeline

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        params = self.params.transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir = config.root_dir,
            data_path = config.data_path,
            tokenizer_name = config.tokenizer_name,
            MAX_INPUT_LENGTH = params.MAX_INPUT_LENGTH,
            MAX_TARGET_LENGTH = params.MAX_TARGET_LENGTH
        )

        return data_transformation_config

In [14]:
import os
from Text_summariser.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

In [19]:
class DataTransformation:

    def __init__(self, config):
        """
        Initialize the Data Transformation component.

        Args:
            config: Configuration object containing tokenizer name,
                    max input/output lengths.
        """
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)

    def run(self):
        dataset = load_from_disk(self.config.data_path)

        tokenized_dataset = self.transform(dataset)

        tokenized_dataset.save_to_disk(
            os.path.join(self.config.root_dir, "samsum_dataset")
        )

    def create_prompt(self, dialogue):
        """
        Convert a dialogue into an instruction-based prompt.
        """

        return f"""Summarize the following conversation.

Dialogue:
{dialogue}

Summary:"""

    def preprocess_function(self, batch):
        """
        Tokenize the dialogue prompts and target summaries.
        """

        prompts = [
            self.create_prompt(dialogue)
            for dialogue in batch["dialogue"]
        ]

        model_inputs = self.tokenizer(
            prompts,
            max_length=self.config.MAX_INPUT_LENGTH,
            truncation=True
        )

        labels = self.tokenizer(
            text_target=batch["summary"],
            max_length=self.config.MAX_TARGET_LENGTH,
            truncation=True
        )

        model_inputs["labels"] = labels["input_ids"]

        return model_inputs

    def transform(self, dataset):
        """
        Apply preprocessing to the entire dataset.
        """

        tokenized_dataset = dataset.map(
            self.preprocess_function,
            batched=True,
            remove_columns=dataset["train"].column_names,
            desc="Tokenizing dataset"
        )

        return tokenized_dataset

In [20]:
try:
    config = configurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.run()
except Exception as e:
    raise e

[2026-07-30 23:00:14,838: INFO: common: ymal file config\config.yaml loaded sucessfully]
[2026-07-30 23:00:14,842: INFO: common: ymal file params.yaml loaded sucessfully]
[2026-07-30 23:00:14,845: INFO: common: created directory at artifacts]
[2026-07-30 23:00:14,847: INFO: common: created directory at artifacts/data_transformation]
[2026-07-30 23:00:15,246: INFO: _client: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/config.json "HTTP/1.1 200 OK"]
[2026-07-30 23:00:15,460: INFO: _client: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"]
[2026-07-30 23:00:15,604: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/t5-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"]
[2026-07-30 23:00:15,757: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google-t5/t5-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTT

Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 90559.59 examples/s] 
